# 42_05 - RQ2/RQ3: Robust aggregation vs serangan + harga ketahanan (Paper 4-2)**Tujuan:** (RQ2) bandingkan 5 aturan agregasi di bawah serangan pada f=20%;(RQ3) ukur 'harga ketahanan' = MCC tiap aturan saat f=0% vs FedAvg.**Sumber tunggal:** `robust_agg.py` + `../aws/fed_node.py`.**Output:** `fed_out42/defense.csv` (RQ2) + `fed_out42/clean_cost.csv` (RQ3).> Jalankan setelah 42_01. JANGAN mengarang angka.

## 0. Setup

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scikit-learn'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.metrics import matthews_corrcoef
AWS_DIR=os.path.abspath(os.path.join('..','aws'))
import importlib.util as _u
_spec=_u.spec_from_file_location('fed_node', os.path.join(AWS_DIR,'fed_node.py'))
fed=_u.module_from_spec(_spec); _spec.loader.exec_module(fed)
sys.path.insert(0, os.path.abspath('.')); import robust_agg as ra
OUTDIR='fed_out42'; os.makedirs(OUTDIR, exist_ok=True)
ROUNDS=60; LOCAL_EPOCHS=3; LR=3e-3; SEED=42
print('setup 42_05 siap')

## 1. Muat partisi + run_fed

In [ ]:
def need(path):
    assert os.path.exists(path),'PRASYARAT GAGAL: %s - jalankan 42_01 dulu'%path
    return path
d=np.load(need(os.path.join(OUTDIR,'parts_K10.npz')),allow_pickle=True)
X=np.asarray(d['X'],float); y=np.asarray(d['y'],int); parts=list(d['parts']); K=int(d['K'])
g=np.load(need(os.path.join(OUTDIR,'global_test.npz')),allow_pickle=True)
Xg=np.asarray(g['X'],float); yg=np.asarray(g['y'],int)
def predict_with(W,mu,sd):
    def f(Z):
        out,_=fed.forward(W, fed.zscore_apply(Z,mu,sd)); return (out>=0.5).astype(int)
    return f
def run_fed(rule='fedavg', attack=None, mal_ids=(), gamma=10.0, bd_frac=0.5,
            rounds=ROUNDS, epochs=LOCAL_EPOCHS, lr=LR, seed=SEED):
    cli=[]
    for k in range(K):
        ii=parts[k]; Xk=X[ii].copy(); yk=y[ii].copy(); rng=np.random.RandomState(seed+k)
        if attack=='label_flip' and k in mal_ids: yk=ra.attack_label_flip(yk,rng)
        if attack=='backdoor'  and k in mal_ids: Xk,yk=ra.poison_backdoor_data(Xk,yk,bd_frac,rng)
        if len(yk): mu,sd=fed.zscore_fit(Xk)
        else: mu,sd=np.zeros(X.shape[1]),np.ones(X.shape[1])
        cli.append(dict(X=fed.zscore_apply(Xk,mu,sd),y=yk,n=len(yk)))
    Wg=fed.init_weights(seed); f_byz=max(1,len(mal_ids)) if mal_ids else 1
    for t in range(rounds):
        wl=[]; sizes=[]
        for k in range(K):
            if cli[k]['n']==0: continue
            Wl=fed.train_local(Wg,cli[k]['X'],cli[k]['y'],epochs=epochs,lr=lr,mu=0.0,W_global=Wg,seed=seed)
            if attack=='scale' and k in mal_ids: Wl=ra.attack_scale_weights(Wl,gamma=gamma)
            wl.append(Wl); sizes.append(cli[k]['n'])
        Wg,_=ra.aggregate(rule,wl,sizes,f=f_byz,beta=max(1,f_byz))
    mu_g,sd_g=fed.zscore_fit(X); pf=predict_with(Wg,mu_g,sd_g); yp=pf(Xg)
    mcc=round(matthews_corrcoef(yg,yp),4)
    asr=round(ra.attack_success_rate(pf,Xg,yg),4) if attack=='backdoor' else None
    return mcc,asr,Wg
RULES=['fedavg','krum','multi_krum','trimmed_mean','median']
print('K',K,'| run_fed siap | rules',RULES)

## 2. RQ2 - 5 aturan agregasi pada f=20% (2 klien jahat) x 3 serangan

In [ ]:
mal=(0,1)   # f=20%
rows=[]
for rule in RULES:
    for atk in ['label_flip','scale','backdoor']:
        mcc,asr,_=run_fed(rule, attack=atk, mal_ids=mal)
        rows.append({'rule':rule,'attack':atk,'MCC':mcc,'ASR':asr})
        print(f'{rule:12s} {atk:11s} -> MCC {mcc} ASR {asr}')
dfd=pd.DataFrame(rows); dfd.to_csv(os.path.join(OUTDIR,'defense.csv'),index=False)
import IPython.display as ipd; ipd.display(dfd.pivot(index='rule',columns='attack',values='MCC'))
print('=== SEL 2 (RQ2 pertahanan) SELESAI ===')

## 3. RQ3 - harga ketahanan (f=0%, tanpa serangan)

In [ ]:
rows2=[]; base=None
for rule in RULES:
    mcc,_,_=run_fed(rule, attack=None, mal_ids=())
    if rule=='fedavg': base=mcc
    rows2.append({'rule':rule,'MCC_clean':mcc})
for r in rows2: r['delta_vs_fedavg']=round(r['MCC_clean']-base,4) if base is not None else None
dfc=pd.DataFrame(rows2); dfc.to_csv(os.path.join(OUTDIR,'clean_cost.csv'),index=False)
import IPython.display as ipd; ipd.display(dfc)
print('CEK: delta kecil = harga ketahanan rendah (robust-agg tak banyak korbankan MCC bersih).')
print('=== SEL 3 (RQ3 harga ketahanan) SELESAI ===')

## 4. Catatan untuk naskah- RQ2 -> Tabel tab:defense; RQ3 -> Tabel tab:clean di `paper-4-2.tex`.- Diharapkan robust-agg >> FedAvg di bawah serangan; delta f=0 kecil. Dilaporkan apa adanya.- Bila robust-agg KELIRU buang klien jujur (heterogenitas lintas-dataset) -> MCC turun;  temuan ini (bila terjadi) dilaporkan jujur sbg kontribusi.